# Cross-Target SHAP Entropy Analysis of Kinase Inhibitor Activity
## A Framework for Quantifying Shared and Target-Specific Structure-Activity Relationships

**Lead Author:** Dr. Abubakar Siddiq Salihu  
**Co-authors:** Muhammad Sulaiman Rahama, Wan Mohd Nuzul Hakimi Wan Salleh  
**Affiliation:** Department of Pure and Industrial Chemistry, Umaru Musa Yar'adua University, Katsina, Nigeria  
**Target Journal:** Molecular Informatics (Wiley, Q2)  
**Repository:** https://github.com/Asidsal/kinase-shap-qsar  

---

## Overview

This notebook implements a complete, reproducible pipeline for cross-target SHAP analysis
of kinase QSAR models. It takes Article 01 output files as input and produces all figures,
tables, and statistics reported in the manuscript.

## Input requirements

The following files from the Article 01 pipeline must be present before running:

| File | Location | Description |
|------|----------|-------------|
|  |  | Curated compound-target pairs |
|  |  | Precomputed Morgan fingerprints (593,518 x 2048) |
|  |  | SMILES-to-fingerprint-row index mapping |
|  |  | Per-target RF model performance metrics |

**Set  in Cell 1 to point to your Article 01 output directory.**

## Notebook structure

| Cell | Description | Est. runtime |
|------|-------------|-------------|
| 1 | Environment setup | <1 min |
| 2 | Load data + precomputed fingerprints | <1 min |
| Fix Cell | Merge target_class + set smiles_col | <1 min |
| 3 | Reconstruct scaffold splits | ~2 min |
| 4 | Retrain RF models + AUROC verification | ~1 min |
| 5 | SHAP computation (TreeExplainer) | ~30 min |
| 6 | Bit-to-fragment mapping | ~1 min |
| 7 | Cross-target Spearman correlation | <1 min |
| 8 | Consensus vs divergent classification | <1 min |
| 9 | SAR validation | <1 min |
| 10 | Contrast target analysis | <1 min |
| 11A-E | Publication figures (Figures 1-5) | <2 min |
| 12 | Results export | <1 min |

## Scope statement

All SHAP attributions derive from a single scaffold-based train-test split per target.
Stability of feature attributions across alternative splits was not assessed. Interpretations
describe chemical patterns associated with model predictions under this specific data
partition and should not be understood as split-invariant chemical rules. The cross-target
aggregation across 31 independently constructed splits provides implicit stability evidence
at the consensus level.

---


---
## CELL 1 — Environment Setup

In [ ]:
# ============================================================
# CELL 1: Environment Setup
# Install shap if not present: !pip install shap
# ============================================================

import os, sys, warnings, pickle, json
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from scipy.stats import spearmanr, pearsonr
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import shap

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ============================================================
# PATHS — edit ART01_DIR to your Article 01 outputs folder
# ============================================================
# ============================================================
# SET THIS PATH to your Article 01 output directory
# ============================================================
ART01_DIR    = Path("../Article_01/article01_outputs")  # <-- edit this
ART01_DATA   = ART01_DIR / "data"

ART03_DIR    = Path("./article03_outputs")
DATA_DIR     = ART03_DIR / "data"
FIGURES_DIR  = ART03_DIR / "figures"
SHAP_DIR     = ART03_DIR / "shap_values"

for d in [ART03_DIR, DATA_DIR, FIGURES_DIR, SHAP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_SEED   = 42
np.random.seed(RANDOM_SEED)

MORGAN_RADIUS = 2
MORGAN_BITS   = 2048
TOP_FEATURES  = 200    # top N bits by mean |SHAP| for cross-target analysis

# RF hyperparameters — identical to Article 01
RF_PARAMS = dict(
    n_estimators     = 300,
    min_samples_leaf = 2,
    class_weight     = 'balanced',
    random_state     = RANDOM_SEED,
    n_jobs           = -1
)

print(f"✅ Environment ready — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"   Python : {sys.version.split()[0]}")
print(f"   shap   : {shap.__version__}")
print(f"   Art01  : {ART01_DIR.resolve()}")
print(f"   Art03  : {ART03_DIR.resolve()}")

---
## CELL 2 — Load Curated Data and Precomputed Fingerprints

In [ ]:
# ============================================================
# CELL 2: Load Article 01 outputs
#   - bindingdb_curated.parquet  → compound-target pairs
#   - morgan_fps.npy             → precomputed fingerprints (all compounds)
#   - smiles_to_idx.pkl          → SMILES → row index in morgan_fps.npy
#   - model_results.csv          → AUROC reference values
# ============================================================

print("Loading Article 01 data...")

# Curated dataset
curated_df = pd.read_parquet(ART01_DATA / 'bindingdb_curated.parquet')
print(f"   Curated dataset  : {len(curated_df):,} rows, {curated_df['uniprot_id'].nunique()} targets")
print(f"   Columns          : {list(curated_df.columns)}")

# Precomputed Morgan fingerprints
morgan_fps = np.load(ART01_DATA / 'morgan_fps.npy')
print(f"   morgan_fps shape : {morgan_fps.shape}")

# SMILES-to-index mapping
with open(ART01_DATA / 'smiles_to_idx.pkl', 'rb') as f:
    smiles_to_idx = pickle.load(f)
print(f"   smiles_to_idx    : {len(smiles_to_idx):,} entries")

# Model results (AUROC reference)
model_results = pd.read_csv(ART01_DATA / 'model_results.csv')
print(f"   model_results    : {len(model_results):,} rows")

# Identify column names — handle variations
# Label column: 'label', 'activity', 'binary_label'
label_col  = next((c for c in curated_df.columns if c in ['label','activity','binary_label']), None)
smiles_col = next((c for c in curated_df.columns if c in ['smiles','canonical_smiles','Ligand SMILES']), None)
uid_col    = next((c for c in curated_df.columns if c in ['uniprot_id','UniProt','target_id']), None)
tname_col  = next((c for c in curated_df.columns if c in ['target_name','Target Name','target']), None)
tclass_col = next((c for c in curated_df.columns if c in ['target_class','class','protein_class']), None)

print(f"\n   Detected columns:")
print(f"     label  : {label_col}")
print(f"     smiles : {smiles_col}")
print(f"     uid    : {uid_col}")
print(f"     tname  : {tname_col}")
print(f"     tclass : {tclass_col}")

assert all([label_col, smiles_col, uid_col, tclass_col]), \
    "❌ Could not detect required columns. Check column names in bindingdb_curated.parquet"

print("\n✅ All data loaded successfully.")

---
## Fix Cell — Merge target_class and set smiles_col

Run this cell immediately after Cell 2. It merges  from 
into the curated dataframe (which does not contain this column) and switches 
to  to match the keys used in .


In [ ]:
# ============================================================
# FIX CELL — Run immediately after Cell 2
# Merges target_class from model_results into curated_df
# and switches smiles_col to smiles_std (matches smiles_to_idx)
# ============================================================

uid_class_map = (
    model_results[["uniprot_id", "target_class"]]
    .drop_duplicates("uniprot_id")
    .set_index("uniprot_id")["target_class"]
    .to_dict()
)

curated_df["target_class"] = curated_df["uniprot_id"].map(uid_class_map)
tclass_col = "target_class"
smiles_col  = "smiles_std"  # matches smiles_to_idx.pkl keys

print(f"target_class added: {curated_df['target_class'].nunique()} unique classes")
print(f"Classes: {sorted(curated_df['target_class'].dropna().unique().tolist())}")
print(f"smiles_col -> '{smiles_col}'")
print(f"label_col  : {label_col}")
print(f"uid_col    : {uid_col}")
print(f"tclass_col : {tclass_col}")


---
## CELL 3 — Extract Kinase Targets and Reconstruct Scaffold Splits

In [ ]:
# ============================================================
# CELL 3: Extract kinase + contrast targets, reconstruct splits
# ============================================================

KINASE_LABEL    = 'Kinase'
CONTRAST_LABELS = ['Transporter', 'Metabolic']  # illustrative only

# RF results only — these are our reference AUROC values
rf_ref = model_results[model_results['model'] == 'RandomForest'].copy()
kinase_ref   = rf_ref[rf_ref['target_class'] == KINASE_LABEL].copy()
contrast_ref = rf_ref[rf_ref['target_class'].isin(CONTRAST_LABELS)].copy()

print(f"Kinase targets in model_results   : {len(kinase_ref)}")
print(f"Contrast targets in model_results : {len(contrast_ref)}")

# Scaffold split function — matches Article 01 Murcko greedy protocol
def get_murcko_scaffold(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except:
        return None

def scaffold_split(df, smiles_col, label_col, test_frac=0.2, seed=42):
    """Greedy Murcko scaffold split — replicates Article 01 protocol."""
    df = df.copy().reset_index(drop=True)
    df['_scaffold'] = df[smiles_col].apply(get_murcko_scaffold)
    df = df.dropna(subset=['_scaffold']).reset_index(drop=True)

    groups = df.groupby('_scaffold').apply(lambda x: x.index.tolist())
    sizes  = groups.apply(len).sort_values(ascending=False)

    n_test = int(len(df) * test_frac)
    test_idx, train_idx = [], []
    for scaffold in sizes.index:
        idxs = groups[scaffold]
        if len(test_idx) < n_test:
            test_idx.extend(idxs)
        else:
            train_idx.extend(idxs)

    return df.loc[train_idx].copy(), df.loc[test_idx].copy()

# Build split data for all targets
split_data = {}   # {uid: {X_train, y_train, X_test, y_test, smiles_test}}
skipped    = []

all_ref = pd.concat([kinase_ref, contrast_ref], ignore_index=True)

for _, row in tqdm(all_ref.iterrows(), total=len(all_ref), desc='Reconstructing splits'):
    uid = row[uid_col] if uid_col in row else row['uniprot_id']

    # Filter curated data to this target
    target_df = curated_df[curated_df[uid_col] == uid][[smiles_col, label_col]].copy()
    target_df = target_df.dropna(subset=[smiles_col, label_col])

    if len(target_df) < 60:
        skipped.append(uid)
        continue

    train_df, test_df = scaffold_split(target_df, smiles_col, label_col,
                                        test_frac=0.2, seed=RANDOM_SEED)

    # Verify both classes present in both partitions
    if (train_df[label_col].nunique() < 2 or test_df[label_col].nunique() < 2):
        skipped.append(uid)
        continue

    # Retrieve fingerprints from precomputed array
    def get_fps(df_part):
        fps, labels, smiles_out = [], [], []
        for _, r in df_part.iterrows():
            smi = r[smiles_col]
            if smi in smiles_to_idx:
                fps.append(morgan_fps[smiles_to_idx[smi]])
            else:
                # Fallback: compute on-the-fly
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, MORGAN_RADIUS, nBits=MORGAN_BITS)
                    fps.append(np.array(fp, dtype=np.uint8))
                else:
                    continue
            labels.append(r[label_col])
            smiles_out.append(smi)
        return np.array(fps, dtype=np.uint8), np.array(labels), smiles_out

    X_train, y_train, _ = get_fps(train_df)
    X_test,  y_test, smiles_test = get_fps(test_df)

    if len(X_train) < 30 or len(X_test) < 10:
        skipped.append(uid)
        continue

    split_data[uid] = {
        'X_train'    : X_train,
        'y_train'    : y_train,
        'X_test'     : X_test,
        'y_test'     : y_test,
        'smiles_test': np.array(smiles_test)
    }

print(f"\n✅ Splits ready: {len(split_data)} targets ({len(skipped)} skipped: {skipped})")

---
## CELL 4 — Retrain RF Models

In [ ]:
# ============================================================
# CELL 4: Retrain RF models and verify AUROC vs Article 01
# ============================================================

rf_models    = {}
auroc_check  = []
AUROC_TOL    = 0.05   # tolerance for retrained models

ref_auroc = dict(zip(
    all_ref['uniprot_id'] if 'uniprot_id' in all_ref.columns else all_ref[uid_col],
    all_ref['AUROC']
))

for uid, sd in tqdm(split_data.items(), desc='Training RF models'):
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(sd['X_train'], sd['y_train'])
    rf_models[uid] = rf

    y_prob = rf.predict_proba(sd['X_test'])[:, 1]
    auroc  = roc_auc_score(sd['y_test'], y_prob)
    ref    = ref_auroc.get(uid, np.nan)
    delta  = auroc - ref

    auroc_check.append({
        'uniprot_id'    : uid,
        'auroc_art01'   : ref,
        'auroc_art03'   : round(auroc, 6),
        'delta'         : round(delta, 6),
        'within_tol'    : abs(delta) <= AUROC_TOL
    })

auroc_df = pd.DataFrame(auroc_check)
auroc_df.to_csv(DATA_DIR / 'auroc_verification.csv', index=False)

failed = auroc_df[~auroc_df['within_tol']]
print(f"\n{'='*55}")
print(f"RF training complete: {len(rf_models)} models")
print(f"AUROC verification (tolerance ±{AUROC_TOL}):")
print(f"   Within tolerance : {auroc_df['within_tol'].sum()} / {len(auroc_df)}")
if len(failed) > 0:
    print(f"   ⚠️  Outside tolerance ({len(failed)}):")
    print(failed[['uniprot_id','auroc_art01','auroc_art03','delta']].to_string(index=False))
    print("   Note: Differences > 0.05 may reflect RF stochasticity or")
    print("   minor curation differences. Review these targets manually.")
else:
    print("   ✅ All models within tolerance.")

print(f"\nMedian AUROC Art01 : {auroc_df['auroc_art01'].median():.4f}")
print(f"Median AUROC Art03 : {auroc_df['auroc_art03'].median():.4f}")

---
## CELL 5 — Compute SHAP Values (TreeExplainer)

In [ ]:
# ============================================================
# CELL 5: SHAP computation using TreeExplainer
# Computes mean |SHAP| per feature per target on test set.
# Kinase targets only at this stage.
# ============================================================

kinase_ids   = kinase_ref['uniprot_id'].tolist()
contrast_ids = contrast_ref['uniprot_id'].tolist()

# Filter to verified kinase targets
kinase_ids_verified = [uid for uid in kinase_ids if uid in rf_models]
print(f"Computing SHAP for {len(kinase_ids_verified)} kinase targets...")
print("Expected time: ~2-5 min depending on target sizes\n")

# shap_matrix: shape (n_kinase_targets, MORGAN_BITS)
# Each row = mean |SHAP| values across test set for one target
shap_matrix  = np.zeros((len(kinase_ids_verified), MORGAN_BITS))
shap_raw     = {}   # {uid: full SHAP value array (n_test, n_bits)}

for i, uid in enumerate(tqdm(kinase_ids_verified, desc='SHAP computation')):
    sd    = split_data[uid]
    model = rf_models[uid]

    explainer  = shap.TreeExplainer(model)
    shap_vals  = explainer.shap_values(sd['X_test'])

    # shap_values returns list [class0, class1] for classifiers
    # Use class 1 (active) SHAP values
    if isinstance(shap_vals, list):
        shap_active = shap_vals[1]   # shape: (n_test, n_bits)
    else:
        shap_active = shap_vals

    mean_abs_shap = np.abs(shap_active).mean(axis=0)  # shape: (n_bits,)
    shap_matrix[i] = mean_abs_shap
    shap_raw[uid]  = shap_active

    # Save raw SHAP per target
    np.save(SHAP_DIR / f"shap_{uid}.npy", shap_active)

# Save mean |SHAP| matrix
shap_matrix_df = pd.DataFrame(
    shap_matrix,
    index   = kinase_ids_verified,
    columns = [f'bit_{i}' for i in range(MORGAN_BITS)]
)
shap_matrix_df.to_csv(DATA_DIR / 'shap_mean_abs_matrix.csv')
np.save(DATA_DIR / 'shap_matrix.npy', shap_matrix)

print(f"\n✅ SHAP computation complete")
print(f"   Matrix shape: {shap_matrix.shape} (targets × bits)")
print(f"   Saved to    : {DATA_DIR / 'shap_mean_abs_matrix.csv'}")

---
## CELL 6 — Bit-to-Fragment Mapping

In [ ]:
# ============================================================
# CELL 6: Map Morgan fingerprint bits to chemical fragments
# Uses RDKit bitInfo to identify which atoms/environments
# each bit encodes across the compound pool.
# ============================================================

from rdkit.Chem import Draw
from IPython.display import display

# Identify top N bits by mean |SHAP| across ALL kinase targets
global_mean_shap = shap_matrix.mean(axis=0)   # shape: (MORGAN_BITS,)
top_bit_indices  = np.argsort(global_mean_shap)[::-1][:TOP_FEATURES]

print(f"Top {TOP_FEATURES} bits by global mean |SHAP| identified.")
print(f"Processing fragment mapping...\n")

# Collect SMILES for all kinase targets (test sets)
all_test_smiles = []
for uid in kinase_ids_verified:
    if split_data[uid]['smiles_test'] is not None:
        all_test_smiles.extend(split_data[uid]['smiles_test'].tolist())

# Deduplicate
all_test_smiles = list(set(all_test_smiles))
print(f"Total unique test-set SMILES across kinase targets: {len(all_test_smiles):,}")

# For each top bit: collect example molecules and atom environments
bit_fragment_map = {}   # {bit_idx: {'smiles_examples': [...], 'description': str}}

# Sample up to 500 molecules for bit mapping (speed)
np.random.seed(RANDOM_SEED)
sample_smiles = np.random.choice(all_test_smiles,
                                  min(500, len(all_test_smiles)),
                                  replace=False).tolist()

# Pre-parse molecules
sample_mols = []
for smi in sample_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        sample_mols.append((smi, mol))

print(f"Parsed {len(sample_mols)} molecules for bit mapping")

for bit_idx in tqdm(top_bit_indices, desc='Mapping bits to fragments'):
    examples = []
    for smi, mol in sample_mols:
        bit_info = {}
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, MORGAN_RADIUS, nBits=MORGAN_BITS, bitInfo=bit_info
        )
        if bit_idx in bit_info:
            examples.append({
                'smiles'   : smi,
                'atom_idx' : bit_info[bit_idx][0][0],
                'radius'   : bit_info[bit_idx][0][1]
            })
        if len(examples) >= 10:   # 10 examples per bit is sufficient
            break

    # Assign coarse fragment class based on the first example atom environment
    frag_class = 'Unknown'
    if examples:
        try:
            mol_ex  = Chem.MolFromSmiles(examples[0]['smiles'])
            atom    = mol_ex.GetAtomWithIdx(examples[0]['atom_idx'])
            symbol  = atom.GetSymbol()
            aromatic = atom.GetIsAromatic()
            in_ring  = atom.IsInRing()
            hbd = atom.GetTotalNumHs() > 0 and symbol in ['N','O']
            hba = symbol in ['N','O','F','S']

            if aromatic and symbol == 'C':
                frag_class = 'Aromatic_C'
            elif aromatic and symbol == 'N':
                frag_class = 'Aromatic_N (heterocycle)'
            elif aromatic and symbol == 'O':
                frag_class = 'Aromatic_O (heterocycle)'
            elif symbol == 'N' and hbd:
                frag_class = 'HBD_N'
            elif symbol == 'O' and hbd:
                frag_class = 'HBD_O'
            elif symbol in ['N','O','F','S'] and hba:
                frag_class = 'HBA'
            elif in_ring and not aromatic:
                frag_class = 'Aliphatic_ring'
            elif symbol == 'C' and not aromatic:
                frag_class = 'Aliphatic_C'
            else:
                frag_class = f'Other_{symbol}'
        except:
            pass

    bit_fragment_map[int(bit_idx)] = {
        'global_mean_shap': float(global_mean_shap[bit_idx]),
        'n_examples'       : len(examples),
        'frag_class'       : frag_class,
        'smiles_examples'  : [e['smiles'] for e in examples[:3]]
    }

# Save fragment map
with open(DATA_DIR / 'bit_fragment_map.json', 'w') as f:
    json.dump(bit_fragment_map, f, indent=2)

# Summary table
frag_df = pd.DataFrame([
    {'bit': k, **v} for k, v in bit_fragment_map.items()
]).sort_values('global_mean_shap', ascending=False)
frag_df = frag_df.drop(columns=['smiles_examples'])
frag_df.to_csv(DATA_DIR / 'fragment_map_summary.csv', index=False)

print(f"\n✅ Fragment mapping complete for {len(bit_fragment_map)} bits")
print(f"\nFragment class distribution (top {TOP_FEATURES} bits):")
print(frag_df['frag_class'].value_counts().to_string())

---
## CELL 7 — Cross-Target SHAP Consistency Analysis

In [ ]:
# ============================================================
# CELL 7: Cross-target SHAP consistency
# Computes pairwise Spearman correlations between per-target
# SHAP profiles restricted to top TOP_FEATURES bits.
# ============================================================

# Restrict to top features
shap_top = shap_matrix[:, top_bit_indices]  # shape: (n_targets, TOP_FEATURES)

n_targets = len(kinase_ids_verified)
corr_matrix = np.zeros((n_targets, n_targets))

print(f"Computing {n_targets}x{n_targets} pairwise Spearman correlation matrix...")
for i in range(n_targets):
    for j in range(n_targets):
        if i == j:
            corr_matrix[i, j] = 1.0
        elif j > i:
            r, _ = spearmanr(shap_top[i], shap_top[j])
            corr_matrix[i, j] = r
            corr_matrix[j, i] = r

corr_df = pd.DataFrame(
    corr_matrix,
    index   = kinase_ids_verified,
    columns = kinase_ids_verified
)
corr_df.to_csv(DATA_DIR / 'shap_spearman_correlation.csv')

# Summary statistics
upper_tri = corr_matrix[np.triu_indices(n_targets, k=1)]
print(f"\nCross-target SHAP profile Spearman correlation (N={n_targets} kinase targets):")
print(f"   Mean    : {upper_tri.mean():.4f}")
print(f"   Median  : {np.median(upper_tri):.4f}")
print(f"   SD      : {upper_tri.std():.4f}")
print(f"   Min     : {upper_tri.min():.4f}")
print(f"   Max     : {upper_tri.max():.4f}")
print(f"   Pairs r>0.5 : {(upper_tri > 0.5).sum()} / {len(upper_tri)}")
print(f"   Pairs r<0.2 : {(upper_tri < 0.2).sum()} / {len(upper_tri)}")

# Hierarchical clustering (Ward, cosine distance) for Figure 2
dist_matrix = pdist(shap_top, metric='cosine')
linkage_matrix = linkage(dist_matrix, method='ward')
np.save(DATA_DIR / 'linkage_matrix.npy', linkage_matrix)

# Get cluster order for heatmap
from scipy.cluster.hierarchy import leaves_list
cluster_order = leaves_list(linkage_matrix)
ordered_ids   = [kinase_ids_verified[i] for i in cluster_order]

pd.Series(ordered_ids, name='uniprot_id').to_csv(
    DATA_DIR / 'cluster_order.csv', index=False
)

print(f"\n✅ Correlation matrix and clustering complete.")

---
## CELL 8 — Consensus vs Divergent Feature Identification

In [ ]:
# ============================================================
# CELL 8: Consensus vs divergent features
# Consensus  = high mean |SHAP| + low entropy of rank distribution
# Divergent  = high mean |SHAP| + high entropy of rank distribution
# ============================================================

from scipy.stats import entropy as scipy_entropy

# For each of the top TOP_FEATURES bits:
# Compute (a) mean |SHAP| across targets
# Compute (b) entropy of normalised |SHAP| rank distribution

feature_stats = []

for bit_pos, bit_idx in enumerate(top_bit_indices):
    shap_vals_per_target = shap_top[:, bit_pos]   # one value per target

    mean_shap = shap_vals_per_target.mean()

    # Entropy of the rank-normalised distribution
    # Normalise to probability distribution (softmax-style)
    if shap_vals_per_target.sum() > 0:
        prob_dist = shap_vals_per_target / shap_vals_per_target.sum()
        # Replace zeros to avoid log(0)
        prob_dist = np.where(prob_dist < 1e-10, 1e-10, prob_dist)
        ent = scipy_entropy(prob_dist)
    else:
        ent = 0.0

    # Max possible entropy for uniform distribution
    max_entropy = np.log(len(kinase_ids_verified))
    norm_entropy = ent / max_entropy if max_entropy > 0 else 0.0

    frag_info = bit_fragment_map.get(int(bit_idx), {})

    feature_stats.append({
        'bit_idx'      : int(bit_idx),
        'mean_shap'    : float(mean_shap),
        'entropy'      : float(ent),
        'norm_entropy' : float(norm_entropy),
        'frag_class'   : frag_info.get('frag_class', 'Unknown'),
        'n_examples'   : frag_info.get('n_examples', 0)
    })

features_df = pd.DataFrame(feature_stats).sort_values('mean_shap', ascending=False)

# Classification thresholds
# Consensus  : top 50% mean_shap AND bottom 33% entropy
# Divergent  : top 50% mean_shap AND top 33% entropy
shap_median   = features_df['mean_shap'].median()
ent_low_thr   = features_df['norm_entropy'].quantile(0.33)
ent_high_thr  = features_df['norm_entropy'].quantile(0.67)

features_df['category'] = 'Intermediate'
features_df.loc[
    (features_df['mean_shap'] >= shap_median) &
    (features_df['norm_entropy'] <= ent_low_thr), 'category'
] = 'Consensus'
features_df.loc[
    (features_df['mean_shap'] >= shap_median) &
    (features_df['norm_entropy'] >= ent_high_thr), 'category'
] = 'Divergent'

features_df.to_csv(DATA_DIR / 'feature_classification.csv', index=False)

consensus_features = features_df[features_df['category'] == 'Consensus'].head(10)
divergent_features = features_df[features_df['category'] == 'Divergent'].head(10)

print("TOP 10 CONSENSUS FEATURES (high importance, low entropy)")
print("="*60)
print(consensus_features[['bit_idx','mean_shap','norm_entropy','frag_class']].to_string(index=False))

print("\nTOP 10 DIVERGENT FEATURES (high importance, high entropy)")
print("="*60)
print(divergent_features[['bit_idx','mean_shap','norm_entropy','frag_class']].to_string(index=False))

print(f"\nCategory counts:")
print(features_df['category'].value_counts().to_string())

---
## CELL 9 — SAR Validation: Fragment Presence vs Activity

In [ ]:
# ============================================================
# CELL 9: SAR validation
# For top 3 consensus features: plot active rate when
# fragment is present vs absent across all kinase test sets.
# This links model explanation → real structure-activity data.
# ============================================================

top3_consensus_bits = consensus_features['bit_idx'].values[:3]

sar_results = []

for bit_idx in top3_consensus_bits:
    present_active, present_total = 0, 0
    absent_active,  absent_total  = 0, 0

    for uid in kinase_ids_verified:
        sd = split_data[uid]
        bit_presence = sd['X_test'][:, bit_idx]   # 0 or 1 for each compound
        labels       = sd['y_test']

        present_mask = bit_presence == 1
        absent_mask  = bit_presence == 0

        present_active += labels[present_mask].sum()
        present_total  += present_mask.sum()
        absent_active  += labels[absent_mask].sum()
        absent_total   += absent_mask.sum()

    frag_info = bit_fragment_map.get(int(bit_idx), {})

    sar_results.append({
        'bit_idx'          : int(bit_idx),
        'frag_class'       : frag_info.get('frag_class', 'Unknown'),
        'active_rate_present': present_active / present_total if present_total > 0 else np.nan,
        'active_rate_absent' : absent_active  / absent_total  if absent_total  > 0 else np.nan,
        'n_present'        : int(present_total),
        'n_absent'         : int(absent_total)
    })

sar_df = pd.DataFrame(sar_results)
sar_df['rate_difference'] = sar_df['active_rate_present'] - sar_df['active_rate_absent']
sar_df.to_csv(DATA_DIR / 'sar_validation.csv', index=False)

print("SAR Validation — Top 3 Consensus Features")
print("="*65)
print(sar_df[['bit_idx','frag_class','active_rate_present',
              'active_rate_absent','rate_difference','n_present']].to_string(index=False))
print("\nInterpretation: positive rate_difference = fragment enriched in actives")

---
## CELL 10 — Contrast Target Analysis (Illustrative Only)

In [ ]:
# ============================================================
# CELL 10: Contrast target SHAP profiles
# Transporter + Metabolic targets — illustrative only.
# NOT used in any statistical comparison.
# N is too small (Transporter n=3, Metabolic n=2) for statistics.
# ============================================================

contrast_ids_verified = [uid for uid in contrast_ids if uid in rf_models]
print(f"Contrast targets available: {len(contrast_ids_verified)}")
print(f"⚠️  These are used for ILLUSTRATION ONLY — not statistical comparison\n")

contrast_shap = {}

for uid in tqdm(contrast_ids_verified, desc='Contrast SHAP'):
    sd    = split_data[uid]
    model = rf_models[uid]

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(sd['X_test'])

    if isinstance(shap_vals, list):
        shap_active = shap_vals[1]
    else:
        shap_active = shap_vals

    mean_abs = np.abs(shap_active).mean(axis=0)
    contrast_shap[uid] = mean_abs

# Compute kinase consensus profile (mean across all kinase targets)
kinase_consensus = shap_matrix.mean(axis=0)   # shape: (MORGAN_BITS,)

# Compare contrast profiles to kinase consensus (top features only)
if contrast_ids_verified:
    print("Spearman correlation of contrast SHAP profiles vs kinase consensus:")
    for uid in contrast_ids_verified:
        r, p = spearmanr(
            kinase_consensus[top_bit_indices],
            contrast_shap[uid][top_bit_indices]
        )
        tclass = contrast_ref[contrast_ref['uniprot_id']==uid]['target_class'].values[0]
        tname  = contrast_ref[contrast_ref['uniprot_id']==uid]['target_name'].values[0]
        auroc  = auroc_check[0]['auroc_art03'] if auroc_check else 'N/A'
        art01_auroc = contrast_ref[contrast_ref['uniprot_id']==uid]['AUROC'].values[0]
        print(f"   {uid} ({tclass}) AUROC={art01_auroc:.3f}  r={r:.3f} p={p:.4f}  {tname[:40]}")

    # Save contrast SHAP
    contrast_df = pd.DataFrame(
        {uid: contrast_shap[uid] for uid in contrast_ids_verified},
    ).T
    contrast_df.columns = [f'bit_{i}' for i in range(MORGAN_BITS)]
    contrast_df.to_csv(DATA_DIR / 'contrast_shap_profiles.csv')
    print(f"\n✅ Contrast SHAP profiles saved.")
else:
    print("No contrast targets with valid split data found.")
    print("This is acceptable — Figure 5 will be omitted or replaced with text note.")

---
## CELL 11 — Publication Figures

In [ ]:
# ============================================================
# CELL 11A: Figure 1 — Study overview + kinase subset summary
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Figure 1. Kinase Subset Overview and AUROC Distribution',
             fontsize=13, fontweight='bold')

# Get target names for display
uid_to_name = dict(zip(kinase_ref['uniprot_id'], kinase_ref['target_name']))
uid_to_auroc = dict(zip(
    [r['uniprot_id'] for r in auroc_check],
    [r['auroc_art03'] for r in auroc_check]
))

sorted_uids  = sorted(kinase_ids_verified, key=lambda u: uid_to_auroc.get(u, 0))
sorted_auroc = [uid_to_auroc.get(u, 0) for u in sorted_uids]
sorted_names = [uid_to_name.get(u, u)[:35] for u in sorted_uids]

colors = ['#2ecc71' if a >= 0.9 else '#f39c12' if a >= 0.8 else '#e74c3c'
          for a in sorted_auroc]

ax = axes[0]
ax.barh(range(len(sorted_uids)), sorted_auroc, color=colors,
        edgecolor='white', linewidth=0.4)
ax.set_yticks(range(len(sorted_uids)))
ax.set_yticklabels(
    [f"{uid} {name}" for uid, name in zip(sorted_uids, sorted_names)],
    fontsize=6.5
)
ax.axvline(0.9, color='black', ls='--', lw=1, alpha=0.5, label='AUROC = 0.9')
ax.set_xlabel('RF AUROC (scaffold split)', fontsize=10)
ax.set_title(f'(a) {len(kinase_ids_verified)} kinase targets', fontsize=10)
ax.set_xlim(0.4, 1.02)
ax.legend(fontsize=8)

for i, a in enumerate(sorted_auroc):
    ax.text(a + 0.002, i, f'{a:.3f}', va='center', fontsize=5.5)

# Panel b: AUROC distribution histogram
ax2 = axes[1]
ax2.hist(sorted_auroc, bins=15, color='#3498db', edgecolor='white',
         linewidth=0.5, alpha=0.85)
ax2.axvline(np.median(sorted_auroc), color='red', ls='--', lw=1.5,
            label=f'Median = {np.median(sorted_auroc):.3f}')
ax2.set_xlabel('RF AUROC (scaffold split)', fontsize=10)
ax2.set_ylabel('Number of targets', fontsize=10)
ax2.set_title('(b) AUROC distribution', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'Figure1_kinase_overview.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'Figure1_kinase_overview.svg', bbox_inches='tight')
plt.show()
print("✅ Figure 1 saved.")

In [ ]:
# ============================================================
# CELL 11B: Figure 2 — SHAP heatmap (targets × top features)
# ============================================================

# Use cluster order from Cell 7
shap_top_ordered = shap_top[cluster_order, :]   # reorder targets by cluster
ordered_ids_short = [uid for uid in ordered_ids]

# Annotate top features with fragment class
top_frag_classes = [
    bit_fragment_map.get(int(top_bit_indices[j]), {}).get('frag_class', '?')[:12]
    for j in range(min(50, TOP_FEATURES))   # show top 50 features
]

fig, ax = plt.subplots(figsize=(18, max(8, len(ordered_ids_short) * 0.35)))
sns.heatmap(
    shap_top_ordered[:, :50],
    ax=ax,
    cmap='YlOrRd',
    xticklabels=[f"bit{top_bit_indices[j]}\n{top_frag_classes[j]}" for j in range(50)],
    yticklabels=ordered_ids_short,
    linewidths=0.1,
    linecolor='white',
    cbar_kws={'label': 'Mean |SHAP|', 'shrink': 0.6}
)
ax.set_title('Figure 2. Mean |SHAP| Values — Top 50 Features Across Kinase Targets\n'
             '(Targets ordered by hierarchical clustering, Ward linkage, cosine distance)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Morgan fingerprint bits (annotated by fragment class)', fontsize=9)
ax.set_ylabel('Kinase target (UniProt ID)', fontsize=9)
ax.tick_params(axis='x', labelsize=6, rotation=90)
ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'Figure2_shap_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'Figure2_shap_heatmap.svg', bbox_inches='tight')
plt.show()
print("✅ Figure 2 saved.")

In [ ]:
# ============================================================
# CELL 11C: Figure 3 — Cross-target Spearman correlation matrix
# ============================================================

# Reorder correlation matrix by cluster order
corr_ordered = corr_matrix[np.ix_(cluster_order, cluster_order)]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.zeros_like(corr_ordered, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = False   # show full matrix

sns.heatmap(
    corr_ordered,
    ax=ax,
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    xticklabels=ordered_ids_short,
    yticklabels=ordered_ids_short,
    linewidths=0.3,
    linecolor='white',
    square=True,
    cbar_kws={'label': 'Spearman r', 'shrink': 0.7}
)
ax.set_title(
    'Figure 3. Pairwise Spearman Correlation of SHAP Profiles Across Kinase Targets\n'
    f'(Top {TOP_FEATURES} features by mean |SHAP|; targets ordered by hierarchical clustering)',
    fontsize=11, fontweight='bold'
)
ax.tick_params(axis='x', labelsize=7, rotation=90)
ax.tick_params(axis='y', labelsize=7)

# Add summary statistics as text
ax.text(0.02, -0.08,
        f'Mean r = {upper_tri.mean():.3f} | Median r = {np.median(upper_tri):.3f} | '
        f'Pairs r>0.5: {(upper_tri>0.5).sum()}/{len(upper_tri)}',
        transform=ax.transAxes, fontsize=8, color='#555555')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'Figure3_shap_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'Figure3_shap_correlation_matrix.svg', bbox_inches='tight')
plt.show()
print("✅ Figure 3 saved.")

In [ ]:
# ============================================================
# CELL 11D: Figure 4 — Consensus vs divergent fragments
# Panel a: top 10 consensus features (bar chart + fragment class)
# Panel b: top 10 divergent features
# Panel c: SAR validation — active rate present vs absent
# ============================================================

fig = plt.figure(figsize=(18, 7))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)
fig.suptitle('Figure 4. Consensus and Divergent Chemical Features in Kinase SHAP Analysis',
             fontsize=12, fontweight='bold')

# Panel a: consensus
ax1 = fig.add_subplot(gs[0])
cons = consensus_features.head(10).copy()
colors_a = sns.color_palette('Greens_d', len(cons))
bars = ax1.barh(range(len(cons)), cons['mean_shap'],
                color=colors_a[::-1], edgecolor='white')
ax1.set_yticks(range(len(cons)))
ax1.set_yticklabels(
    [f"bit {row['bit_idx']}\n({row['frag_class'][:15]})" for _, row in cons.iterrows()],
    fontsize=7.5
)
ax1.set_xlabel('Mean |SHAP|', fontsize=9)
ax1.set_title('(a) Top 10 Consensus Features\n(high importance, low entropy)', fontsize=9)
ax1.invert_yaxis()

# Panel b: divergent
ax2 = fig.add_subplot(gs[1])
div = divergent_features.head(10).copy()
colors_b = sns.color_palette('Oranges_d', len(div))
ax2.barh(range(len(div)), div['mean_shap'],
         color=colors_b[::-1], edgecolor='white')
ax2.set_yticks(range(len(div)))
ax2.set_yticklabels(
    [f"bit {row['bit_idx']}\n({row['frag_class'][:15]})" for _, row in div.iterrows()],
    fontsize=7.5
)
ax2.set_xlabel('Mean |SHAP|', fontsize=9)
ax2.set_title('(b) Top 10 Divergent Features\n(high importance, high entropy)', fontsize=9)
ax2.invert_yaxis()

# Panel c: SAR validation
ax3 = fig.add_subplot(gs[2])
x_labels  = [f"bit {row['bit_idx']}\n{row['frag_class'][:12]}" for _, row in sar_df.iterrows()]
x_pos     = np.arange(len(sar_df))
w         = 0.35
ax3.bar(x_pos - w/2, sar_df['active_rate_present'], w,
        label='Fragment present', color='#2ecc71', edgecolor='white')
ax3.bar(x_pos + w/2, sar_df['active_rate_absent'],  w,
        label='Fragment absent',  color='#95a5a6', edgecolor='white')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(x_labels, fontsize=8)
ax3.set_ylabel('Active compound rate', fontsize=9)
ax3.set_title('(c) SAR Validation\n(top 3 consensus features)', fontsize=9)
ax3.set_ylim(0, 1.1)
ax3.legend(fontsize=8)
ax3.axhline(0.934, color='red', ls=':', lw=1, alpha=0.6, label='Dataset baseline (93.4%)')

for i, row in sar_df.iterrows():
    ax3.text(i - w/2, row['active_rate_present'] + 0.02,
             f"{row['active_rate_present']:.2f}", ha='center', fontsize=7)
    ax3.text(i + w/2, row['active_rate_absent']  + 0.02,
             f"{row['active_rate_absent']:.2f}",  ha='center', fontsize=7)

plt.savefig(FIGURES_DIR / 'Figure4_consensus_divergent.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'Figure4_consensus_divergent.svg', bbox_inches='tight')
plt.show()
print("✅ Figure 4 saved.")

In [ ]:
# ============================================================
# CELL 11E: Figure 5 — Contrast target SHAP profiles
# Illustrative only — NOT used for statistical comparison.
# ============================================================

if contrast_ids_verified:
    fig, axes = plt.subplots(1, len(contrast_ids_verified) + 1,
                              figsize=(5 * (len(contrast_ids_verified) + 1), 5))
    fig.suptitle(
        'Figure 5. Contrast: Kinase Consensus vs Low-Predictability Target SHAP Profiles\n'
        '(Illustrative only — Transporter/Metabolic N too small for statistical comparison)',
        fontsize=10, fontweight='bold'
    )

    # Panel 0: kinase consensus profile
    ax = axes[0]
    top50_consensus = kinase_consensus[top_bit_indices[:50]]
    ax.bar(range(50), top50_consensus, color='#2ecc71', alpha=0.8, width=0.8)
    ax.set_xlabel('Feature rank (top 50)', fontsize=8)
    ax.set_ylabel('Mean |SHAP|', fontsize=8)
    ax.set_title(f'Kinase consensus\n(N={len(kinase_ids_verified)} targets)', fontsize=9)
    ax.set_xticks([])

    # Contrast panels
    for i, uid in enumerate(contrast_ids_verified):
        ax = axes[i + 1]
        profile = contrast_shap[uid][top_bit_indices[:50]]
        ax.bar(range(50), profile, color='#e74c3c', alpha=0.8, width=0.8)

        r, _ = spearmanr(kinase_consensus[top_bit_indices], contrast_shap[uid][top_bit_indices])
        tclass = contrast_ref[contrast_ref['uniprot_id']==uid]['target_class'].values[0]
        art01_auroc = contrast_ref[contrast_ref['uniprot_id']==uid]['AUROC'].values[0]
        tname  = contrast_ref[contrast_ref['uniprot_id']==uid]['target_name'].values[0][:30]

        ax.set_xlabel('Feature rank (top 50)', fontsize=8)
        ax.set_ylabel('Mean |SHAP|', fontsize=8)
        ax.set_title(f'{tclass}: {uid}\nAUROC={art01_auroc:.3f}  r={r:.3f} vs kinase',
                     fontsize=8)
        ax.set_xticks([])

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'Figure5_contrast_profiles.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / 'Figure5_contrast_profiles.svg', bbox_inches='tight')
    plt.show()
    print("✅ Figure 5 saved.")
else:
    print("No verified contrast targets — Figure 5 skipped.")
    print("Note for manuscript: mention in text that Transporter/Metabolic targets")
    print("had insufficient data for split reconstruction and are discussed qualitatively.")

---
## CELL 12 — Results Export for Manuscript

In [ ]:
# ============================================================
# CELL 12: Export all results for manuscript
# ============================================================

# Table 1: Kinase target summary
table1 = pd.DataFrame([
    {
        'uniprot_id'  : uid,
        'target_name' : kinase_ref[kinase_ref['uniprot_id']==uid]['target_name'].values[0]
                        if uid in kinase_ref['uniprot_id'].values else uid,
        'n_train'     : split_data[uid]['X_train'].shape[0],
        'n_test'      : split_data[uid]['X_test'].shape[0],
        'AUROC_art03' : uid_to_auroc.get(uid, np.nan)
    }
    for uid in kinase_ids_verified
]).sort_values('AUROC_art03', ascending=False)
table1.to_csv(DATA_DIR / 'Table1_kinase_targets.csv', index=False)

# Table 2: Top consensus features
table2 = consensus_features[['bit_idx','mean_shap','norm_entropy','frag_class']].copy()
table2.to_csv(DATA_DIR / 'Table2_consensus_features.csv', index=False)

# Table 3: Top divergent features
table3 = divergent_features[['bit_idx','mean_shap','norm_entropy','frag_class']].copy()
table3.to_csv(DATA_DIR / 'Table3_divergent_features.csv', index=False)

# Manuscript statistics summary
ms_stats = {
    'n_kinase_targets'         : len(kinase_ids_verified),
    'median_auroc'             : float(np.median([uid_to_auroc.get(u,0) for u in kinase_ids_verified])),
    'mean_auroc'               : float(np.mean([uid_to_auroc.get(u,0) for u in kinase_ids_verified])),
    'top_features_analysed'    : TOP_FEATURES,
    'n_consensus_features'     : int((features_df['category']=='Consensus').sum()),
    'n_divergent_features'     : int((features_df['category']=='Divergent').sum()),
    'cross_target_corr_mean'   : float(upper_tri.mean()),
    'cross_target_corr_median' : float(np.median(upper_tri)),
    'cross_target_corr_sd'     : float(upper_tri.std()),
    'pairs_r_gt_0.5'           : int((upper_tri > 0.5).sum()),
    'pairs_r_lt_0.2'           : int((upper_tri < 0.2).sum()),
    'n_contrast_targets'       : len(contrast_ids_verified),
    'scope_statement'          : 'Single scaffold split per target; SHAP attributions are split-specific',
    'generated'                : datetime.now().isoformat()
}

with open(DATA_DIR / 'manuscript_statistics.json', 'w') as f:
    json.dump(ms_stats, f, indent=2)

print("MANUSCRIPT STATISTICS SUMMARY")
print("="*55)
for k, v in ms_stats.items():
    if k not in ['scope_statement', 'generated']:
        print(f"   {k:<35}: {v}")

print(f"\n{'='*55}")
print(f"ALL OUTPUTS SAVED TO: {ART03_DIR.resolve()}")
print(f"{'='*55}")
print(f"   data/Table1_kinase_targets.csv")
print(f"   data/Table2_consensus_features.csv")
print(f"   data/Table3_divergent_features.csv")
print(f"   data/shap_mean_abs_matrix.csv")
print(f"   data/shap_spearman_correlation.csv")
print(f"   data/feature_classification.csv")
print(f"   data/sar_validation.csv")
print(f"   data/manuscript_statistics.json")
print(f"   figures/Figure1_kinase_overview.png/svg")
print(f"   figures/Figure2_shap_heatmap.png/svg")
print(f"   figures/Figure3_shap_correlation_matrix.png/svg")
print(f"   figures/Figure4_consensus_divergent.png/svg")
print(f"   figures/Figure5_contrast_profiles.png/svg")
print(f"   shap_values/shap_{{uniprot_id}}.npy (per target)")
print(f"\n✅ Article 03 notebook complete.")